# Turkiye Dergi Satis Tahminleri - LightGBM + TUIK Hibrit Ensemble

## Proje Ozeti
Bu notebook'ta Turkiye'de satilan **90 derginin** 2019-2025 gercek satis verilerini kullanarak
2026 ve 2027 yillari icin **sevk, iade ve net satis** tahminleri uretiyoruz.

### Yontem
- **Global LightGBM**: 90 dergi tek modelde birlestirdi (630+ satir)
- **TUIK Entegrasyonu**: Ulusal tiraj trendi ve kategori degisim oranlari ozellik olarak eklendi
- **Walk-Forward Cross Validation**: Gercekci model degerlendirmesi
- **Hibrit Ensemble**: %55 LightGBM + %45 TUIK formulu

### Tahmin Edilen Degiskenler
| Degisken | Aciklama |
|---|---|
| `sevk` | Bayiye gonderilen adet |
| `iade` | Satılamayıp donen adet |
| `net` | Gercek satis = sevk - iade |

In [ ]:
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import openpyxl
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='darkgrid')

print('Kutuphaneler yuklendi')

## 1. Veri Yukleme

In [ ]:
matches = glob.glob('/kaggle/input/**/data.xlsx', recursive=True)
EXCEL_PATH = matches[0] if matches else '/kaggle/input/dergi-tahmin/data.xlsx'
print(f'Excel yolu: {EXCEL_PATH}')

CATEGORY_MAP = {
    'BAYAN':   ['COSMOPOLITAN','ELLE','VOGUE','INSTYLE','MARIE CLAIRE','BAZAAR','JOY','OLIVIA',
                'ELELE','BURDA','STYLE','ALEM','MADAME'],
    'HABER':   ['NEWSWEEK','ECONOMIST','FOREIGN','TIME','ATLAS','NATIONAL GEOGRAPHIC',
                'GEO','FOCUS','BILIM','POPULAR'],
    'ERKEK':   ['GQ','MEN','ESQUIRE','MAXIM','PLAYBOY','FHM','MOTOR'],
    'COCUK':   ['MINIK','YARAMAZ','COCUK','TOYZZ','WINX','DISNEY','MANGA','POGO'],
    'EGLENCE': ['TV','SINEMA','MUZIK','SPORT','FUTBOL','BASKET','CEPTE','DIGITAL','PC','CHIP'],
    'EV':      ['ELLE DECO','HOME','INTERIOR','MUTFAK','GARDEN','YEMEK','CUISINE'],
    'IS':      ['CAPITAL','FORBES','PARA','EKONOMIST','BUSINESS','FORTUNE','BRAND'],
}

def assign_category(name):
    nu = name.upper()
    for cat, keywords in CATEGORY_MAP.items():
        if any(k in nu for k in keywords):
            return cat
    return 'DIGER'

print('Kategori haritasi hazir')

In [ ]:
def load_tuik_data(wb):
    nat_tiraj = {}
    cat_pct = {}
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            continue
        header = [str(c).strip() if c else '' for c in rows[0]]
        if any('ulusal' in h.lower() or 'tiraj' in h.lower() for h in header):
            for row in rows[1:]:
                if row[0] and str(row[0]).strip().isdigit():
                    yr = int(str(row[0]).strip())
                    val = row[1] if len(row) > 1 else None
                    if val:
                        try:
                            nat_tiraj[yr] = float(str(val).replace(',','.'))
                        except:
                            pass
        if any('kategori' in h.lower() or 'pct' in h.lower() or '%' in h for h in header):
            for row in rows[1:]:
                if row[0] and row[1]:
                    try:
                        cat_pct[str(row[0]).strip()] = float(str(row[1]).replace(',','.'))
                    except:
                        pass
    DEFAULT_NAT = {2019:100,2020:92,2021:86,2022:79,2023:73,2024:68,2025:63,2026:59,2027:56}
    if not nat_tiraj:
        nat_tiraj = DEFAULT_NAT
    for yr in range(2019, 2028):
        if yr not in nat_tiraj:
            nat_tiraj[yr] = DEFAULT_NAT.get(yr, 56)
    return nat_tiraj, cat_pct


def load_magazine_data(wb):
    ws = None
    for name in wb.sheetnames:
        if 'veri' in name.lower() or 'data' in name.lower() or 'set' in name.lower():
            ws = wb[name]
            break
    if ws is None:
        ws = wb[wb.sheetnames[0]]
    magazines = {}
    for row in ws.iter_rows(values_only=True):
        if not row[0]:
            continue
        cells = [str(c).strip() if c is not None else '' for c in row]
        year_match = re.search(r'20(1[89]|2[0-7])', cells[0])
        if not year_match:
            continue
        yr = int(year_match.group())
        if yr > 2025:
            continue
        mag_name = cells[1] if len(cells) > 1 else ''
        if not mag_name or mag_name in ('None', ''):
            continue
        def to_f(v):
            try:
                return float(str(v).replace('.','').replace(',','.'))
            except:
                return 0.0
        sevk = to_f(cells[2]) if len(cells) > 2 else 0
        iade = to_f(cells[3]) if len(cells) > 3 else 0
        net  = to_f(cells[4]) if len(cells) > 4 else sevk - iade
        if mag_name not in magazines:
            magazines[mag_name] = {'years':[],'sevk':[],'iade':[],'net':[]}
        if yr not in magazines[mag_name]['years']:
            magazines[mag_name]['years'].append(yr)
            magazines[mag_name]['sevk'].append(sevk)
            magazines[mag_name]['iade'].append(iade)
            magazines[mag_name]['net'].append(net)
    return magazines


wb = openpyxl.load_workbook(EXCEL_PATH, read_only=True, data_only=True)
nat_tiraj, cat_pct = load_tuik_data(wb)
raw_mags = load_magazine_data(wb)

print(f'Toplam dergi: {len(raw_mags)}')
print(f'TUIK yillari: {sorted(nat_tiraj.keys())}')

## 2. Kesifsel Veri Analizi (EDA)

In [ ]:
records = []
for mag, d in raw_mags.items():
    cat = assign_category(mag)
    for i, yr in enumerate(d['years']):
        records.append({'dergi':mag,'kategori':cat,'yil':yr,
                        'sevk':d['sevk'][i],'iade':d['iade'][i],'net':d['net'][i]})
df = pd.DataFrame(records)
print(df.shape)
df.head(10)

In [ ]:
yearly = df.groupby('yil')['net'].sum().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].fill_between(yearly['yil'], yearly['net'], alpha=0.3, color='#4f8ef7')
axes[0].plot(yearly['yil'], yearly['net'], 'o-', color='#4f8ef7', linewidth=2.5)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
axes[0].set_title('Tum Dergiler - Toplam Net Satis (2019-2025)', fontsize=13, fontweight='bold')

cat_counts = df.groupby('kategori')['dergi'].nunique().sort_values()
colors = plt.cm.tab10(np.linspace(0, 1, len(cat_counts)))
axes[1].barh(cat_counts.index, cat_counts.values, color=colors)
axes[1].set_title('Kategori Basina Dergi Sayisi', fontsize=13, fontweight='bold')
for i, v in enumerate(cat_counts.values):
    axes[1].text(v+0.1, i, str(v), va='center')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top15 = df[df['yil']==2025].nlargest(15,'net')[['dergi','kategori','net']].reset_index(drop=True)
colors2 = plt.cm.viridis(np.linspace(0.3, 0.9, len(top15)))
plt.figure(figsize=(14,6))
plt.barh(top15['dergi'][::-1], top15['net'][::-1], color=colors2[::-1])
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.title('2025 En Cok Satan 15 Dergi - Net Satis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('top15_2025.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ozellik Muhendisligi (Feature Engineering)

| Ozellik | Aciklama |
|---|---|
| `prev_net / prev_sevk / prev_iade` | t-1 degerleri (lag-1) |
| `lag2_net / lag2_sevk` | t-2 degerleri (lag-2) |
| `net_cagr3 / sevk_cagr3` | 3 yillik bilesik buyume orani |
| `iade_ratio` | Iade / Sevk orani |
| `nat_tiraj` | TUIK ulusal tiraj endeksi |
| `nat_ratio` | Yillik ulusal tiraj degisim orani |
| `cat_ratio` | Kategorinin yillik degisim orani |
| `hybrid_ratio` | nat_ratio x cat_ratio |
| `cat_enc` | Kategori (Label Encoded) |

In [ ]:
CAT_TREND = {'BAYAN':-0.04,'HABER':-0.03,'ERKEK':-0.05,'COCUK':-0.02,
             'EGLENCE':-0.04,'EV':-0.03,'IS':-0.02,'DIGER':-0.04}

le = LabelEncoder()
all_cats = sorted(set(assign_category(m) for m in raw_mags))
le.fit(all_cats)

FEATURES = ['cat_enc','nat_tiraj','cat_pct_val','nat_ratio','cat_ratio','hybrid_ratio',
            'prev_net','prev_sevk','prev_iade','lag2_net','lag2_sevk',
            'iade_ratio','net_cagr3','sevk_cagr3']

def build_global_df():
    rows_net, rows_sevk, rows_iade = [], [], []
    for mag, d in raw_mags.items():
        cat = assign_category(mag)
        cat_enc = int(le.transform([cat])[0])
        years = sorted(d['years'])
        idx_map = {y:i for i,y in enumerate(d['years'])}
        f_nets  = [d['net'][idx_map[y]]  for y in years]
        f_sevks = [d['sevk'][idx_map[y]] for y in years]
        f_iades = [d['iade'][idx_map[y]] for y in years]
        for i, yr in enumerate(years):
            if i < 2:
                continue
            prev_net  = f_nets[i-1]
            prev_sevk = f_sevks[i-1]
            prev_iade = f_iades[i-1]
            lag2_net  = f_nets[i-2]
            lag2_sevk = f_sevks[i-2]
            nt = nat_tiraj.get(yr, 63)
            nt_prev = nat_tiraj.get(yr-1, 63)
            nat_ratio = nt / nt_prev if nt_prev else 1.0
            cat_pct_val = cat_pct.get(cat, CAT_TREND.get(cat, -0.04))
            cat_ratio = 1.0 + cat_pct_val
            hybrid_ratio = nat_ratio * cat_ratio
            iade_ratio = prev_iade / prev_sevk if prev_sevk > 0 else 0.3
            if i >= 3 and f_nets[i-3] > 0 and prev_net >= 0:
                net_cagr3 = float((prev_net / f_nets[i-3]) ** (1/3) - 1)
            else:
                net_cagr3 = 0.0
            if i >= 3 and f_sevks[i-3] > 0 and prev_sevk >= 0:
                sevk_cagr3 = float((prev_sevk / f_sevks[i-3]) ** (1/3) - 1)
            else:
                sevk_cagr3 = 0.0
            feat = [cat_enc,nt,cat_pct_val,nat_ratio,cat_ratio,hybrid_ratio,
                    prev_net,prev_sevk,prev_iade,lag2_net,lag2_sevk,
                    iade_ratio,net_cagr3,sevk_cagr3]
            rows_net.append(feat  + [f_nets[i]])
            rows_sevk.append(feat + [f_sevks[i]])
            rows_iade.append(feat + [f_iades[i]])
    cols = FEATURES + ['target']
    return (pd.DataFrame(rows_net, columns=cols),
            pd.DataFrame(rows_sevk,columns=cols),
            pd.DataFrame(rows_iade,columns=cols))

df_net, df_sevk, df_iade = build_global_df()
print(f'Global DataFrame boyutu: {df_net.shape}')
df_net[FEATURES].describe().round(3)

## 4. Walk-Forward Cross Validation

```
Fold 1: Egitim: 2019-2022  -> Test: 2023
Fold 2: Egitim: 2019-2023  -> Test: 2024
Fold 3: Egitim: 2019-2024  -> Test: 2025
```

In [ ]:
LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    max_depth=6, min_child_samples=5,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1
)

def walk_forward_cv(df_all):
    test_years_nat = [73, 68, 63]
    maes, mapes, r2s = [], [], []
    for nt_test in test_years_nat:
        nt_prev = nt_test + 5
        train_mask = df_all['nat_tiraj'] > nt_test
        test_mask  = (df_all['nat_tiraj'] <= nt_test) & (df_all['nat_tiraj'] >= nt_prev - 5)
        if train_mask.sum() < 10 or test_mask.sum() < 1:
            continue
        X_tr = df_all.loc[train_mask, FEATURES]
        y_tr = df_all.loc[train_mask, 'target']
        X_te = df_all.loc[test_mask,  FEATURES]
        y_te = df_all.loc[test_mask,  'target']
        m = lgb.LGBMRegressor(**LGB_PARAMS)
        m.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(30,verbose=False), lgb.log_evaluation(-1)])
        preds = m.predict(X_te)
        mae  = mean_absolute_error(y_te, preds)
        r2   = r2_score(y_te, preds)
        valid = y_te[y_te > 100]
        mape = np.mean(np.abs((valid - preds[y_te > 100]) / valid)) * 100 if len(valid) > 0 else np.nan
        maes.append(mae); mapes.append(mape); r2s.append(r2)
        print(f'  Fold nat_tiraj={nt_test}: MAE={mae:.0f}  MAPE={mape:.1f}%  R2={r2:.3f}')
    return {'mae':np.mean(maes),'mape':np.nanmean(mapes),'r2':np.mean(r2s)}

print('=== Walk-Forward CV ===')
cv_metrics = walk_forward_cv(df_net)
print(f'Ortalama: MAE={cv_metrics["mae"]:.0f}  MAPE={cv_metrics["mape"]:.1f}%  R2={cv_metrics["r2"]:.3f}')

## 5. Final Model Egitimi

In [ ]:
def train_model(df_all):
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(df_all[FEATURES], df_all['target'], callbacks=[lgb.log_evaluation(-1)])
    return m

print('Modeller egitiliyor...')
model_net  = train_model(df_net)
model_sevk = train_model(df_sevk)
model_iade = train_model(df_iade)
global_mae = cv_metrics['mae']
print(f'Modeller egitildi  (Global MAE: {global_mae:.0f})')

In [ ]:
fi = pd.DataFrame({'feature':FEATURES,'importance':model_net.feature_importances_})
fi = fi.sort_values('importance')
colors3 = plt.cm.plasma(np.linspace(0.3, 0.9, len(fi)))
plt.figure(figsize=(10,6))
plt.barh(fi['feature'], fi['importance'], color=colors3)
plt.title('LightGBM Ozellik Onemi - Net Satis Modeli', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 2026-2027 Tahminleri

**Ensemble Formulu:**
```
Final = 0.55 x LightGBM + 0.45 x TUIK_hibrit
Guven Araligi: +-1 x MAE
```

In [ ]:
def forecast_magazine(mag, d):
    cat = assign_category(mag)
    cat_enc = int(le.transform([cat])[0])
    years = sorted(d['years'])
    idx_map = {y:i for i,y in enumerate(d['years'])}
    nets  = [d['net'][idx_map[y]]  for y in years]
    sevks = [d['sevk'][idx_map[y]] for y in years]
    iades = [d['iade'][idx_map[y]] for y in years]
    results = {}
    cur_nets, cur_sevks, cur_iades = nets.copy(), sevks.copy(), iades.copy()
    for pred_yr in [2026, 2027]:
        if len(cur_nets) < 2:
            results[pred_yr] = None
            continue
        prev_net  = cur_nets[-1]
        prev_sevk = cur_sevks[-1]
        prev_iade = cur_iades[-1]
        lag2_net  = cur_nets[-2]
        lag2_sevk = cur_sevks[-2]
        nt = nat_tiraj.get(pred_yr, 59)
        nt_prev = nat_tiraj.get(pred_yr-1, 63)
        nat_ratio = nt / nt_prev if nt_prev else 1.0
        cat_pct_val = cat_pct.get(cat, CAT_TREND.get(cat, -0.04))
        cat_ratio = 1.0 + cat_pct_val
        hybrid_ratio = nat_ratio * cat_ratio
        iade_ratio = prev_iade / prev_sevk if prev_sevk > 0 else 0.3
        n = len(cur_nets)
        if n >= 3 and cur_nets[-3] > 0 and prev_net >= 0:
            net_cagr3 = float((prev_net / cur_nets[-3]) ** (1/3) - 1)
        else:
            net_cagr3 = 0.0
        if n >= 3 and cur_sevks[-3] > 0 and prev_sevk >= 0:
            sevk_cagr3 = float((prev_sevk / cur_sevks[-3]) ** (1/3) - 1)
        else:
            sevk_cagr3 = 0.0
        feat = [[cat_enc,nt,cat_pct_val,nat_ratio,cat_ratio,hybrid_ratio,
                 prev_net,prev_sevk,prev_iade,lag2_net,lag2_sevk,
                 iade_ratio,net_cagr3,sevk_cagr3]]
        lgb_net  = max(float(model_net.predict(feat)[0]),  0)
        lgb_sevk = max(float(model_sevk.predict(feat)[0]), 0)
        lgb_iade = max(float(model_iade.predict(feat)[0]), 0)
        hybrid_net  = max(prev_net  * hybrid_ratio, 0)
        hybrid_sevk = max(prev_sevk * hybrid_ratio, 0)
        hybrid_iade = max(prev_iade * hybrid_ratio, 0)
        final_net  = max(round(0.55*lgb_net  + 0.45*hybrid_net,  2), 0)
        final_sevk = max(round(0.55*lgb_sevk + 0.45*hybrid_sevk, 2), 0)
        final_iade = max(round(0.55*lgb_iade + 0.45*hybrid_iade, 2), 0)
        results[pred_yr] = {
            'net':final_net,'sevk':final_sevk,'iade':final_iade,
            'ci_low':max(final_net - global_mae, 0),
            'ci_high':final_net + global_mae
        }
        cur_nets.append(final_net)
        cur_sevks.append(final_sevk)
        cur_iades.append(final_iade)
    return results

forecasts = {mag: forecast_magazine(mag, d) for mag, d in raw_mags.items()}
print(f'{len(forecasts)} dergi icin tahmin tamamlandi')

## 7. Sonuclar

In [ ]:
result_rows = []
for mag, fc in forecasts.items():
    d = raw_mags[mag]
    last_net = d['net'][-1] if d['net'] else 0
    f26 = fc.get(2026)
    f27 = fc.get(2027)
    result_rows.append({
        'Dergi':mag,
        'Kategori':assign_category(mag),
        'Net_2025':last_net,
        'Net_Tahmin_2026':round(f26['net']) if f26 else None,
        'Net_Tahmin_2027':round(f27['net']) if f27 else None,
        'CI_Alt_2026':round(f26['ci_low']) if f26 else None,
        'CI_Ust_2026':round(f26['ci_high']) if f26 else None,
    })

results_df = pd.DataFrame(result_rows).sort_values('Net_Tahmin_2026', ascending=False)
results_df.head(20)

In [ ]:
top5 = results_df.head(5)['Dergi'].tolist()
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, mag in zip(axes, top5):
    d = raw_mags[mag]
    fc = forecasts[mag]
    split = len(d['years'])
    years_all = d['years'] + [2026, 2027]
    f26 = fc.get(2026)
    f27 = fc.get(2027)
    net_all = d['net'] + [f26['net'] if f26 else None, f27['net'] if f27 else None]
    ax.plot(years_all[:split], net_all[:split], 'o-', color='#4f8ef7', linewidth=2, label='Gercek')
    ax.plot(years_all[split-1:], net_all[split-1:], 's--', color='#f7844f', linewidth=2, label='Tahmin')
    if f26 and f27:
        ax.fill_between([2026,2027],[f26['ci_low'],f27['ci_low']],[f26['ci_high'],f27['ci_high']],alpha=0.2,color='#f7844f')
    ax.set_title(mag[:15], fontsize=9, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
    ax.legend(fontsize=7)
plt.suptitle('Top 5 Dergi - Gercek vs Tahmin (2019-2027)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('top5_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
yearly_total = df.groupby('yil')['net'].sum()
t26 = results_df['Net_Tahmin_2026'].sum()
t27 = results_df['Net_Tahmin_2027'].sum()
yrs  = list(yearly_total.index) + [2026, 2027]
vals = list(yearly_total.values) + [t26, t27]
colors_bar = ['#4f8ef7'] * len(yearly_total) + ['#f7844f','#f7844f']
plt.figure(figsize=(12,5))
bars = plt.bar(yrs, vals, color=colors_bar, alpha=0.85, edgecolor='white')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.title('Tum Dergiler - Toplam Net Satis (2019-2027)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, vals):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
             f'{val/1000:.0f}K', ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('total_market.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'2025: {yearly_total.get(2025,0):>10,.0f}')
print(f'2026: {t26:>10,.0f}  ({(t26/yearly_total.get(2025,1)-1)*100:+.1f}%)')
print(f'2027: {t27:>10,.0f}  ({(t27/yearly_total.get(2025,1)-1)*100:+.1f}%)')

In [ ]:
results_df.to_csv('dergi_tahminleri_2026_2027.csv', index=False, encoding='utf-8-sig')
print('CSV kaydedildi: dergi_tahminleri_2026_2027.csv')
print(f'Model Performansi (Walk-Forward CV):')
print(f'  MAE  : {cv_metrics["mae"]:>10,.0f}')
print(f'  MAPE : {cv_metrics["mape"]:>10.1f}%')
print(f'  R2   : {cv_metrics["r2"]:>10.3f}')